In [ ]:
import mlflow
import os
from dotenv import load_dotenv

from mlflow.genai import scorer


load_dotenv()

os.environ["MLFLOW_TRACKING_URI"] = "http://localhost:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "log_model_examples"

In [2]:
prompt = """
You are a helpful assistant that can classify news articles into one of the following categories:
- World
- Sports
- Business
- Science
Article: {article}
"""

initial_prompt = mlflow.genai.register_prompt(
    name="news_classifier",
    template=prompt,
)

2025/12/02 10:39:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: news_classifier, version 1


In [3]:
%%writefile lc_model.py

import mlflow
from mlflow.pyfunc import PythonModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from mlflow.models import set_model


class LangchainModel(PythonModel):
    def __init__(self):
        super().__init__()

    def load_context(self, context):
        print("Loading context")
        rate_limiter = InMemoryRateLimiter(
            requests_per_second=0.1,
            check_every_n_seconds=0.1,
            max_bucket_size=10,
        )
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash-lite",
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
            rate_limiter=rate_limiter,
        )

    def predict(self, model_input: dict) -> str:
        prompt_template = mlflow.genai.load_prompt(
            "news_classifier", version=1
        ).template
        prompt = prompt_template.format(article=model_input)
        response = self.llm.invoke(prompt)
        print(response)
        return response.content


set_model(LangchainModel())


Overwriting lc_model.py


In [4]:
with mlflow.start_run(run_name="langchain_model"):
    model_info = mlflow.pyfunc.log_model(
        name="lc_model",
        python_model="lc_model.py",
    )

d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\pyfunc\model.py:185: UserWarning: Type hint used in the model's predict function is not supported for MLflow's schema validation. Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[dict]` for a valid signature. Remove the type hint to disable this warning. To enable validation for the input data, specify input example or model signature when logging the model. 
  func_info = _get_func_info_if_type_hint_supported(predict_attr)
d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\pyfunc\__init__.py:3212: UserWarning: Failed to infer signature from type hint: Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[dict]` for a valid signature.
  signature_from_type_hints = _infer_signature_from_type_hints(
2025/12/0

🏃 View run langchain_model at: http://localhost:5000/#/experiments/360306633431671969/runs/d6a56bf56db545ad8a7e27f892dbbe83
🧪 View experiment at: http://localhost:5000/#/experiments/360306633431671969


In [6]:
model_info.model_uri

'models:/m-da5a1a6fbf8d454286b10cc1b3f3bdbb'

In [ ]:
# Load AG News dataset from Hugging Face as pandas dataframe
from datasets import load_dataset

dataset = load_dataset("ag_news", split="train")
df = dataset.to_pandas()

df["label"] = df["label"].map({0: "World", 1: "Sports", 2: "Business", 3: "Science"})

df = df.sample(frac=1).reset_index(drop=True)


NUM_SAMPLES = 20
train_data = []
for i in range(NUM_SAMPLES):
    article = df.iloc[i]["text"]
    expected = df.iloc[i]["label"]
    eval_dict = {
        "inputs": {"article": article},
        "expectations": {"expected_response": expected},
    }
    train_data.append(eval_dict)


d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
model = mlflow.pyfunc.load_model(model_uri=model_info.model_uri)

Loading context


d:\youtube\TheAIGuy\NLP\mlflow_examples\.venv\Lib\site-packages\mlflow\pyfunc\model.py:185: UserWarning: Type hint used in the model's predict function is not supported for MLflow's schema validation. Type hints must be wrapped in list[...] because MLflow assumes the predict method to take multiple input instances. Specify your type hint as `list[dict]` for a valid signature. Remove the type hint to disable this warning. To enable validation for the input data, specify input example or model signature when logging the model. 
  func_info = _get_func_info_if_type_hint_supported(predict_attr)


In [7]:
def predict_fn(article):
    response = model.predict(article)
    return response

In [8]:
@scorer
def exact_match(outputs, expectations):
    expectations = expectations["expected_response"]
    return outputs == expectations


with mlflow.start_run(run_name="evaluation"):
    results = mlflow.genai.evaluate(
        data=train_data,
        scorers=[exact_match],
        predict_fn=predict_fn,
        model_id=model.model_id,
    )

2025/12/02 10:45:08 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


content='Sports' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--73f3090d-80e4-4420-8b2b-f005c2763acd-0' usage_metadata={'input_tokens': 76, 'output_tokens': 1, 'total_tokens': 77, 'input_token_details': {'cache_read': 0}}


2025/12/02 10:45:21 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-1a4949913e8045539bcfb9befe7248c5
2025/12/02 10:45:21 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
Evaluating:   0%|          | 0/20 [Elapsed: 00:00, Remaining: ?] 

content='Sports' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--5d8289c5-69c6-4419-956f-91fc547faa6f-0' usage_metadata={'input_tokens': 91, 'output_tokens': 1, 'total_tokens': 92, 'input_token_details': {'cache_read': 0}}


Evaluating:   5%|▌         | 1/20 [Elapsed: 00:13, Remaining: 04:16] 

...

content='Business' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--7332dcd0-0eec-4000-a28c-e8f7c4cf23ac-0' usage_metadata={'input_tokens': 101, 'output_tokens': 1, 'total_tokens': 102, 'input_token_details': {'cache_read': 0}}


Evaluating:  95%|█████████▌| 19/20 [Elapsed: 03:09, Remaining: 00:09] 

content='Sports' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'grounding_metadata': {}, 'model_provider': 'google_genai'} id='lc_run--a2137bde-94c8-45a8-ab12-eaa9522875a3-0' usage_metadata={'input_tokens': 70, 'output_tokens': 1, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}}


Evaluating: 100%|██████████| 20/20 [Elapsed: 03:19, Remaining: 00:00] 


[Trace(trace_id=tr-56b78ff138a0a2f4f313a67e29bfe370), Trace(trace_id=tr-155765040eda107e26f0f0d90dcf63d8), Trace(trace_id=tr-6a53f91ce57cc1fe93c428da88a6a4ad), Trace(trace_id=tr-3a1859bd174e1d87410e919ae0ebf7d7), Trace(trace_id=tr-19597bdc0563fbe5e8e91410a7a62f00), Trace(trace_id=tr-dfa0499c33b9952725697de3f8a750ac), Trace(trace_id=tr-f967b1956eb0c6b77c2470e1fdb75d00), Trace(trace_id=tr-176928bff322ef6d1138efa51f6046fc), Trace(trace_id=tr-e86efd0caf2c8f53036fda0b7f3f7565), Trace(trace_id=tr-51533ee552939ada3d08800a635b6d84)]